# AdaLi vs Sigmoid Comparison (MNIST, CIFAR-10, CIFAR-100)

This notebook compares **AdaLi** and **Sigmoid** surrogate activations from this repository using:
- Datasets: MNIST, CIFAR-10, CIFAR-100
- Architectures: a simple CNN and a small ResNet-style model

In [39]:
import os
import copy
import time
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

from config import AdaLiConfig
from modules.sgAdaLi import AdaLi
from modules.sgSigmoid import Sigmoid
from modules.neuron import IF

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)

Torch: 2.11.0+cpu
Torchvision: 0.26.0+cpu


In [40]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Keep a stable copy of the original AdaLi config values.
BASE_ADALI_CFG = copy.deepcopy(AdaLiConfig)
BASE_ADALI_CFG

Device: cpu


{'Epoch': 500, 'epoch': 1, 'Vth': 1.0, 'Left': [1.5, 0.3], 'Right': [1.5, 0.3]}

In [41]:
@dataclass
class ExperimentConfig:
    data_root: str = './data'
    batch_size: int = 128
    num_workers: int = 0
    epochs: int = 3
    lr: float = 1e-3
    weight_decay: float = 1e-4

    # Quick mode: limit dataset size for faster prototyping.
    train_subset: int | None = 20000
    test_subset: int | None = 5000

    # SNN temporal parameters
    timesteps: int = 1

    adali_alpha: float = 0.5
    adali_beta: float = 0.5
    adali_left: float = 1.5
    adali_right: float = 1.5

    sigmoid_alpha: float = 4.0

cfg = ExperimentConfig()
cfg

ExperimentConfig(data_root='./data', batch_size=128, num_workers=0, epochs=3, lr=0.001, weight_decay=0.0001, train_subset=20000, test_subset=5000, timesteps=1, adali_alpha=0.5, adali_beta=0.5, adali_left=1.5, adali_right=1.5, sigmoid_alpha=4.0)

In [42]:
def _maybe_subset(dataset, n, seed=SEED):
    if n is None or n >= len(dataset):
        return dataset
    g = torch.Generator()
    g.manual_seed(seed)
    idx = torch.randperm(len(dataset), generator=g)[:n].tolist()
    return Subset(dataset, idx)


def _build_dataset(dataset_cls, root, train, transform, retries=3, base_wait_s=3):
    import urllib.error

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return dataset_cls(root, train=train, download=True, transform=transform)
        except (urllib.error.HTTPError, urllib.error.URLError) as err:
            last_err = err
            if attempt == retries:
                break
            wait_s = base_wait_s * attempt
            print(
                f"Download issue for {dataset_cls.__name__} (train={train}) - "
                f"attempt {attempt}/{retries}: {err}. Retrying in {wait_s}s..."
            )
            time.sleep(wait_s)

    # If files are already present locally, this succeeds without network.
    try:
        print(f"Using local cache for {dataset_cls.__name__} (train={train}).")
        return dataset_cls(root, train=train, download=False, transform=transform)
    except Exception as cache_err:
        raise RuntimeError(
            f"Failed to prepare {dataset_cls.__name__} (train={train}). "
            f"Last download error: {last_err}. Cache error: {cache_err}"
        ) from last_err


def get_dataloaders(dataset_name: str, cfg: ExperimentConfig):
    dataset_name = dataset_name.lower()

    if dataset_name == 'mnist':
        transform_train = transforms.Compose([
            transforms.RandomCrop(28, padding=2),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        train_set = _build_dataset(torchvision.datasets.MNIST, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.MNIST, cfg.data_root, train=False, transform=transform_test)
        in_channels = 1
        num_classes = 10

    elif dataset_name == 'cifar10':
        mean = (0.4914, 0.4822, 0.4465)
        std = (0.2470, 0.2435, 0.2616)
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_set = _build_dataset(torchvision.datasets.CIFAR10, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.CIFAR10, cfg.data_root, train=False, transform=transform_test)
        in_channels = 3
        num_classes = 10

    elif dataset_name == 'cifar100':
        mean = (0.5071, 0.4867, 0.4408)
        std = (0.2675, 0.2565, 0.2761)
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_set = _build_dataset(torchvision.datasets.CIFAR100, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.CIFAR100, cfg.data_root, train=False, transform=transform_test)
        in_channels = 3
        num_classes = 100

    else:
        raise ValueError(f'Unsupported dataset: {dataset_name}')

    train_set = _maybe_subset(train_set, cfg.train_subset)
    test_set = _maybe_subset(test_set, cfg.test_subset)

    train_loader = DataLoader(
        train_set,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, test_loader, in_channels, num_classes

In [43]:
from modules.neuron import IF

def set_surrogate_mode(name: str, cfg: ExperimentConfig):
    # Reset global config first to avoid cross-run side effects.
    AdaLiConfig['Vth'] = BASE_ADALI_CFG['Vth']
    AdaLiConfig['Left'][0] = BASE_ADALI_CFG['Left'][0]
    AdaLiConfig['Right'][0] = BASE_ADALI_CFG['Right'][0]

    if name.lower() == 'adali':
        AdaLiConfig['Left'][0] = cfg.adali_left
        AdaLiConfig['Right'][0] = cfg.adali_right
    elif name.lower() == 'sigmoid':
        # Sigmoid constructor sets Left/Right to inf internally.
        pass
    else:
        raise ValueError(f'Unsupported surrogate: {name}')


def get_surrogate_function(name: str, cfg: ExperimentConfig):
    """Return a surrogate function class (not instance) for use in neurons."""
    name = name.lower()
    
    if name == 'adali':
        class AdaLiSurrogate:
            def __init__(self):
                self.alpha = cfg.adali_alpha
                self.beta = cfg.adali_beta
            def __call__(self):
                return AdaLi(alpha=self.alpha, beta=self.beta)
        return AdaLiSurrogate()
    elif name == 'sigmoid':
        class SigmoidSurrogate:
            def __init__(self):
                self.alpha = cfg.sigmoid_alpha
            def __call__(self):
                return Sigmoid(alpha=self.alpha)
        return SigmoidSurrogate()
    else:
        raise ValueError(f'Unsupported surrogate: {name}')

In [44]:
class SpikingSimpleCNN(nn.Module):
    """SNN with temporal spiking neurons using IF neurons."""
    def __init__(self, in_channels: int, num_classes: int, surrogate_fn, timesteps: int = 4):
        super().__init__()
        self.timesteps = timesteps
        
        # Conv layer 1
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.neuron1 = IF(surrogate_function=surrogate_fn)
        self.pool1 = nn.MaxPool2d(2)
        
        # Conv layer 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(64)
        self.neuron2 = IF(surrogate_function=surrogate_fn)
        self.pool2 = nn.MaxPool2d(2)
        
        # Conv layer 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(128)
        self.neuron3 = IF(surrogate_function=surrogate_fn)
        self.pool3 = nn.AdaptiveAvgPool2d((1, 1))
        
        self.classifier = nn.Linear(128, num_classes)
    
    def forward(self, x):
        # x shape: (batch, channels, height, width)
        batch_size = x.shape[0]
        outputs = []
        
        # Process each timestep sequentially
        for t in range(self.timesteps):
            # Reset neuron states at the start of each timestep
            self.neuron1.reset()
            self.neuron2.reset()
            self.neuron3.reset()
            
            # Layer 1
            x_t = self.conv1(x)
            x_t = self.bn1(x_t)
            x_t = self.neuron1(x_t)
            x_t = self.pool1(x_t)
            
            # Layer 2
            x_t = self.conv2(x_t)
            x_t = self.bn2(x_t)
            x_t = self.neuron2(x_t)
            x_t = self.pool2(x_t)
            
            # Layer 3
            x_t = self.conv3(x_t)
            x_t = self.bn3(x_t)
            x_t = self.neuron3(x_t)
            x_t = self.pool3(x_t)
            
            # Classifier
            x_t = torch.flatten(x_t, 1)
            x_t = self.classifier(x_t)
            outputs.append(x_t)
        
        # Average outputs across timesteps
        x = torch.stack(outputs, dim=1).mean(dim=1)  # (batch, num_classes)
        return x


class SpikingBasicBlock(nn.Module):
    """Basic block for spiking ResNet."""
    def __init__(self, in_ch: int, out_ch: int, stride: int, surrogate_fn):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.neuron1 = IF(surrogate_function=surrogate_fn)
        
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.neuron2 = IF(surrogate_function=surrogate_fn)
        
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
    
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.neuron1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + identity
        out = self.neuron2(out)
        return out


class SpikingSmallResNet(nn.Module):
    """SNN ResNet with temporal spiking neurons."""
    def __init__(self, in_channels: int, num_classes: int, surrogate_fn, timesteps: int = 4):
        super().__init__()
        self.timesteps = timesteps
        
        self.stem_conv = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.stem_bn = nn.BatchNorm2d(32)
        self.stem_neuron = IF(surrogate_function=surrogate_fn)
        
        self.layer1 = nn.Sequential(
            SpikingBasicBlock(32, 32, stride=1, surrogate_fn=surrogate_fn),
            SpikingBasicBlock(32, 32, stride=1, surrogate_fn=surrogate_fn),
        )
        self.layer2 = nn.Sequential(
            SpikingBasicBlock(32, 64, stride=2, surrogate_fn=surrogate_fn),
            SpikingBasicBlock(64, 64, stride=1, surrogate_fn=surrogate_fn),
        )
        self.layer3 = nn.Sequential(
            SpikingBasicBlock(64, 128, stride=2, surrogate_fn=surrogate_fn),
            SpikingBasicBlock(128, 128, stride=1, surrogate_fn=surrogate_fn),
        )
        
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)
    
    def _reset_neurons(self):
        """Recursively reset all IF neurons."""
        for module in self.modules():
            if isinstance(module, IF):
                module.reset()
    
    def forward(self, x):
        batch_size = x.shape[0]
        outputs = []
        
        # Process each timestep sequentially
        for t in range(self.timesteps):
            # Reset all neuron states
            self._reset_neurons()
            
            # Stem
            x_t = self.stem_conv(x)
            x_t = self.stem_bn(x_t)
            x_t = self.stem_neuron(x_t)
            
            # ResNet layers
            x_t = self.layer1(x_t)
            x_t = self.layer2(x_t)
            x_t = self.layer3(x_t)
            
            # Classification
            x_t = self.pool(x_t)
            x_t = torch.flatten(x_t, 1)
            x_t = self.fc(x_t)
            outputs.append(x_t)
        
        # Average outputs across timesteps
        x = torch.stack(outputs, dim=1).mean(dim=1)
        return x


def build_model(model_name: str, in_channels: int, num_classes: int, surrogate_fn, timesteps: int):
    name = model_name.lower()
    if name == 'simple_cnn':
        return SpikingSimpleCNN(in_channels=in_channels, num_classes=num_classes, 
                               surrogate_fn=surrogate_fn, timesteps=timesteps)
    if name == 'small_resnet':
        return SpikingSmallResNet(in_channels=in_channels, num_classes=num_classes, 
                                 surrogate_fn=surrogate_fn, timesteps=timesteps)
    raise ValueError(f'Unsupported model: {model_name}')

In [45]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


def run_experiment(dataset_name: str, model_name: str, surrogate_name: str, cfg: ExperimentConfig):
    set_surrogate_mode(surrogate_name, cfg)

    train_loader, test_loader, in_channels, num_classes = get_dataloaders(dataset_name, cfg)
    surrogate_fn = get_surrogate_function(surrogate_name, cfg)

    model = build_model(model_name, in_channels=in_channels, num_classes=num_classes, 
                       surrogate_fn=surrogate_fn, timesteps=cfg.timesteps).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    history = []
    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
        })

        print(f'[{dataset_name:8s}] [{model_name:11s}] [{surrogate_name:7s}] Epoch {epoch:2d}/{cfg.epochs} | '
              f'train_acc={train_acc:.4f} test_acc={test_acc:.4f}')

    elapsed = time.time() - start_time
    best_test_acc = max(h['test_acc'] for h in history)

    return {
        'dataset': dataset_name,
        'model': model_name,
        'surrogate': surrogate_name,
        'epochs': cfg.epochs,
        'timesteps': cfg.timesteps,
        'best_test_acc': best_test_acc,
        'final_test_acc': history[-1]['test_acc'],
        'train_subset': cfg.train_subset,
        'test_subset': cfg.test_subset,
        'seconds': elapsed,
        'history': history,
    }

In [46]:
# Reset experiments to retry with fixed architecture
all_runs = []
done = set()
failed_runs = []
print("Experiment state reset. Ready to run benchmark grid.")

Experiment state reset. Ready to run benchmark grid.


In [47]:
# Main benchmark grid
datasets = ['mnist','cifar10', 'cifar100']  
models = ['simple_cnn', 'small_resnet']
surrogates = ['adali', 'sigmoid']

# Keep prior progress if this cell is re-run.
if 'all_runs' not in globals() or not isinstance(all_runs, list):
    all_runs = []

done = {(r['dataset'], r['model'], r['surrogate']) for r in all_runs}
failed_runs = []

# Preflight each dataset once to avoid repeated retries per model/surrogate pair.
ready_datasets = []
for ds in datasets:
    try:
        _ = get_dataloaders(ds, cfg)
        ready_datasets.append(ds)
        print(f'Dataset ready: {ds}')
    except Exception as err:
        print(f'Skipping dataset {ds} for now (data unavailable): {err}')

for ds in ready_datasets:
    for model_name in models:
        for surr in surrogates:
            key = (ds, model_name, surr)
            if key in done:
                print(f'Skipping already completed: {key}')
                continue

            try:
                result = run_experiment(ds, model_name, surr, cfg)
                all_runs.append(result)
                done.add(key)
            except Exception as err:
                msg = str(err)
                failed_runs.append({'dataset': ds, 'model': model_name, 'surrogate': surr, 'error': msg})
                print(f'Failed {key}: {msg}')

print(f'Total completed runs: {len(all_runs)}')
if failed_runs:
    print(f'Failed runs in this pass: {len(failed_runs)}')
    for fr in failed_runs:
        print(f"- ({fr['dataset']}, {fr['model']}, {fr['surrogate']})")

Dataset ready: mnist
Dataset ready: cifar10
Dataset ready: cifar100
[mnist   ] [simple_cnn ] [adali  ] Epoch  1/3 | train_acc=0.1108 test_acc=0.1344
[mnist   ] [simple_cnn ] [adali  ] Epoch  2/3 | train_acc=0.1174 test_acc=0.1180
[mnist   ] [simple_cnn ] [adali  ] Epoch  3/3 | train_acc=0.1290 test_acc=0.1476
[mnist   ] [simple_cnn ] [sigmoid] Epoch  1/3 | train_acc=0.1096 test_acc=0.1114
[mnist   ] [simple_cnn ] [sigmoid] Epoch  2/3 | train_acc=0.1136 test_acc=0.1110
[mnist   ] [simple_cnn ] [sigmoid] Epoch  3/3 | train_acc=0.1172 test_acc=0.1482
[mnist   ] [small_resnet] [adali  ] Epoch  1/3 | train_acc=0.1026 test_acc=0.1156
[mnist   ] [small_resnet] [adali  ] Epoch  2/3 | train_acc=0.1007 test_acc=0.1066


KeyboardInterrupt: 

In [ ]:
records = []
for r in all_runs:
    records.append({
        'dataset': r['dataset'],
        'model': r['model'],
        'surrogate': r['surrogate'],
        'best_test_acc': r['best_test_acc'],
        'final_test_acc': r['final_test_acc'],
        'seconds': r['seconds'],
        'epochs': r['epochs'],
        'train_subset': r['train_subset'],
        'test_subset': r['test_subset'],
    })

df = pd.DataFrame(records).sort_values(['dataset', 'model', 'surrogate']).reset_index(drop=True)
df

In [ ]:
pivot = df.pivot_table(
    index=['dataset', 'model'],
    columns='surrogate',
    values='best_test_acc',
    aggfunc='max'
).reset_index()

if {'adali', 'sigmoid'}.issubset(set(pivot.columns)):
    pivot['adali_minus_sigmoid'] = pivot['adali'] - pivot['sigmoid']

pivot

In [ ]:
# Visual comparison
plot_df = df.copy()
plot_df['label'] = plot_df['dataset'] + ' | ' + plot_df['model']

plt.figure(figsize=(12, 5))
for surrogate_name, grp in plot_df.groupby('surrogate'):
    x = np.arange(len(grp))

# Keep deterministic order across groups
plot_df = plot_df.sort_values(['dataset', 'model', 'surrogate']).reset_index(drop=True)
labels = sorted(plot_df['label'].unique())
x = np.arange(len(labels))
width = 0.35

vals_adali = []
vals_sigmoid = []
for lbl in labels:
    vals_adali.append(float(plot_df[(plot_df['label'] == lbl) & (plot_df['surrogate'] == 'adali')]['best_test_acc'].iloc[0]))
    vals_sigmoid.append(float(plot_df[(plot_df['label'] == lbl) & (plot_df['surrogate'] == 'sigmoid')]['best_test_acc'].iloc[0]))

plt.bar(x - width / 2, vals_adali, width=width, label='AdaLi')
plt.bar(x + width / 2, vals_sigmoid, width=width, label='Sigmoid')
plt.xticks(x, labels, rotation=30, ha='right')
plt.ylabel('Best Test Accuracy')
plt.title('AdaLi vs Sigmoid across Dataset/Architecture')
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save result table for reporting
out_path = 'adali_vs_sigmoid_results.csv'
df.to_csv(out_path, index=False)
print('Saved:', out_path)